In [12]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
import json
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("EXCHANGE_RATE_API_KEY")

In [5]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """This function fetches the currency conversion factor between a given base currency and a target currency."""
    url = f"https://v6.exchangerate-api.com/v6/{API_KEY}/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()

In [6]:
get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1789516801,
 'time_last_update_utc': 'Wed, 16 Sep 2026 00:00:01 +0000',
 'time_next_update_unix': 1789603201,
 'time_next_update_utc': 'Thu, 17 Sep 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.0008}

In [7]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def convert(base_currency_value: int,
            conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """Given a currency conversion rate, this function calculates the target currency value from a given base currency value."""
    return base_currency_value * conversion_rate

In [8]:
convert.invoke({"base_currency_value": 10, "conversion_rate": 96.0008})

960.008

In [13]:
model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [14]:
llm_with_tools = model.bind_tools([get_conversion_factor,convert])

In [15]:
messages = [HumanMessage("What is the conversion factor between USD and INR,"
"and based on that can you convert 10 USD to INR?")]

ai_message = llm_with_tools.invoke(messages)
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_157909',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_157910',
  'type': 'tool_call'}]